# Similarity Mejoras

## Libraries

In [1]:
import sys

print('Python version: ', sys.version)

Python version:  3.11.4 | packaged by conda-forge | (main, Jun 10 2023, 18:08:17) [GCC 12.2.0]


In [2]:
# Desactiva por completo la dependencia con TorchVision en Transformers
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"   # <- clave
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"      # opcional, menos ruido
os.environ["TRANSFORMERS_NO_TF"] = "1"       # <— evita que Transformers intente cargar TensorFlow
os.environ["TRANSFORMERS_NO_FLAX"] = "1"     # opcional, por si acaso

In [3]:
import site, sys
sys.path.insert(0, site.getusersitepackages())
print("usersite:", site.getusersitepackages())

usersite: /home/jovyan/.local/lib/python3.11/site-packages


In [4]:
!pip uninstall -y sentence-transformers transformers tokenizers torchvision

Found existing installation: sentence-transformers 2.6.1
Uninstalling sentence-transformers-2.6.1:
  Successfully uninstalled sentence-transformers-2.6.1
Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3


In [5]:
# Instala versiones probadas para Python 3.11 (sin tocar torch/cuda del sistema)
# NOTA: usamos --user para no requerir root; --no-cache-dir por si hay ruedas viejas cacheadas
!pip install --user --upgrade --no-cache-dir \
  "numpy==1.23.5" \
  "tokenizers==0.20.3" \
  "transformers==4.46.3" \
  "sentence-transformers==2.6.1" \
  "scikit-learn==1.5.2" \
  "pandas==2.2.3" \
  "pyarrow==17.0.0" \
  "numexpr>=2.10.1" \
  "bottleneck>=1.4.0"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 54.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 89.5 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 35.0 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import numpy, pandas, sklearn, transformers, tokenizers, sentence_transformers, torch, pyarrow, numexpr, bottleneck
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| cuda ver:", torch.version.cuda)
print("pyarrow:", pyarrow.__version__)
print("numexpr:", numexpr.__version__)
print("bottleneck:", bottleneck.__version__)

numpy: 1.23.5
pandas: 2.2.3
sklearn: 1.5.2
transformers: 4.46.3 | tokenizers: 0.20.3
sentence-transformers: 2.6.1
torch: 2.8.0+cu128 | cuda: True | cuda ver: 12.8
pyarrow: 17.0.0
numexpr: 2.11.0
bottleneck: 1.5.0


In [7]:
import types

if "torchvision" not in sys.modules:
    tv = types.ModuleType("torchvision")
    tv_t = types.ModuleType("torchvision.transforms")
    class _InterpolationMode: pass
    tv_t.InterpolationMode = _InterpolationMode
    tv.transforms = tv_t
    sys.modules["torchvision"] = tv
    sys.modules["torchvision.transforms"] = tv_t

# (Opcional) Si por alguna razón Transformers todavía cree que hay TF, fuerza a falso:
try:
    import transformers.utils.import_utils as _iu
    _iu.is_tf_available = lambda: False
except Exception:
    pass

print("Guardia OK. usersite first:", sys.path[0])

Guardia OK. usersite first: /home/jovyan/.local/lib/python3.11/site-packages


In [8]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

emb = model.encode(["hola mundo"], convert_to_tensor=True)
print(emb.shape, emb.device)


torch.Size([1, 384]) cuda:0


## A) Setup básico y rutas portables 

In [9]:
import os, re, math, string, random
import numpy as np
import pandas as pd
import torch

# Semillas reproducibles
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# Base portátil: directorio del notebook (si existiese env var BASE, úsala)
BASE = os.environ.get("BASE", os.getcwd())
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [10]:
from pprint import pprint

# Carga DWAs
df_onet = pd.read_csv(os.path.join(DATA_DIR, "dwas.csv"))
onetLabels = df_onet["dwa_title"].astype(str).tolist()

pprint(onetLabels)
print(len(onetLabels))

['Train personnel on proper operational procedures.',
 'Prepare procedural documents.',
 'Recommend technical design or process changes to improve efficiency, '
 'quality, or performance.',
 'Confer with technical personnel to prepare designs or operational plans.',
 'Communicate technical information to suppliers, contractors, or regulatory '
 'agencies.',
 'Design medical devices or appliances.',
 'Research engineering aspects of biological or chemical processes.',
 'Devise research or testing protocols.',
 'Develop operational methods or processes that use green materials or '
 'emphasize sustainability.',
 'Develop technical methods or processes.',
 'Create models of engineering designs or methods.',
 'Maintain operational records or records systems.',
 'Supervise engineering or other technical personnel.',
 'Estimate operational costs.',
 'Estimate time requirements for development or production projects.',
 'Prepare detailed work plans.',
 'Prepare technical reports for internal 

## B) Utils: limpieza y funciones auxiliares

In [11]:
import re, string

In [12]:
BASIC_STOP = {
    "and","or","to","of","the","a","an","with","for","in","on","by","from",
    "at","as","into","than","that","this","those","these","is","are","be"
}

_punct_tbl = str.maketrans({c:" " for c in string.punctuation})

def normalize_text(s: str) -> str:
    s = s.lower().translate(_punct_tbl)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s: str):
    return [t for t in normalize_text(s).split() if t]

def jaccard_stopwords(a: str, b: str, stop=BASIC_STOP):
    A = {t for t in tokenize(a) if t in stop}
    B = {t for t in tokenize(b) if t in stop}
    if not A and not B: return 0.0
    return len(A & B) / len(A | B)

def length_penalty(a: str, b: str):
    la, lb = len(tokenize(a)), len(tokenize(b))
    if max(la, lb) == 0: return 0.0
    return abs(la - lb) / max(la, lb)


In [13]:
import math
from collections import Counter, defaultdict

def split_clauses(text: str) -> list:
    """
    Divide una DWA en micro-acciones por conectores frecuentes (' and ', ' or ', ',', ';').
    Filtra cláusulas muy cortas (<=2 tokens significativos) y normaliza.
    """
    if not isinstance(text, str):
        return []
    s = normalize_text(text)
    # división suave; orden importa: primero ' and ', luego ' or '
    parts = []
    for chunk in re.split(r"[;,]", s):
        for sub in re.split(r"\band\b|\bor\b", chunk):
            sub = sub.strip()
            toks = [t for t in sub.split() if t and t not in BASIC_STOP]
            if len(toks) >= 3:
                parts.append(" ".join(toks))
    if not parts:
        # fallback: una sola cláusula con tokens significativos
        toks = [t for t in s.split() if t and t not in BASIC_STOP]
        return [" ".join(toks)] if toks else []
    return parts

def minmax_norm(x, xmin=None, xmax=None):
    """Normalización min-max robusta. Si rango ~0, devuelve 0."""
    x = float(x)
    if xmin is None or xmax is None or xmax - xmin < 1e-12:
        return 0.0
    return (x - xmin) / (xmax - xmin + 1e-12)

# ----------------------------- BM25 minimal (sin librerías externas) -----------------------------
class BM25Index:
    """
    Implementación simple de BM25 para corpus pequeño/mediano.
    - docs_tokens: lista de listas de tokens (ya normalizados) del corpus
    - k1, b: hiperparámetros BM25 estándar
    """
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.docs = docs_tokens
        self.N = len(docs_tokens)
        self.k1 = k1
        self.b = b
        self.avgdl = sum(len(d) for d in docs_tokens) / max(1, self.N)
        # DF y IDF
        self.df = Counter()
        for d in docs_tokens:
            for t in set(d):
                self.df[t] += 1
        self.idf = {t: math.log(1 + (self.N - df + 0.5) / (df + 0.5)) for t, df in self.df.items()}
        # longitud documentos
        self.doc_len = [len(d) for d in docs_tokens]
        # TF por doc
        self.tf = [Counter(d) for d in docs_tokens]

    def score_query(self, q_tokens):
        """Devuelve lista de scores BM25 (longitud = N) para la query tokenizada."""
        scores = np.zeros(self.N, dtype=np.float32)
        for i in range(self.N):
            s = 0.0
            K = self.k1 * ((1 - self.b) + self.b * self.doc_len[i] / max(1e-12, self.avgdl))
            tfi = self.tf[i]
            for t in q_tokens:
                if t not in self.idf: 
                    continue
                tf = tfi.get(t, 0)
                if tf <= 0: 
                    continue
                s += self.idf[t] * (tf * (self.k1 + 1)) / (tf + K)
            scores[i] = s
        return scores

def build_bm25_index(texts: list) -> BM25Index:
    """Construye el índice BM25 con los textos normalizados/tokenizados (sin stopwords)."""
    docs_tokens = []
    for s in texts:
        s_norm = normalize_text(str(s))
        toks = [t for t in s_norm.split() if t and t not in BASIC_STOP]
        docs_tokens.append(toks)
    return BM25Index(docs_tokens)

def bm25_query_scores(bm25: BM25Index, query_text: str, candidate_idx: np.ndarray) -> np.ndarray:
    """
    Devuelve BM25 normalizado (0–1) para la query y sólo para los índices candidatos.
    Normaliza por min-max sobre los candidatos (robusto).
    """
    q_norm = normalize_text(str(query_text))
    q_tokens = [t for t in q_norm.split() if t and t not in BASIC_STOP]
    raw_scores = bm25.score_query(q_tokens)
    if candidate_idx.size == 0:
        return np.zeros(0, dtype=np.float32)
    vals = raw_scores[candidate_idx]
    mn, mx = float(np.min(vals)), float(np.max(vals))
    normed = np.array([minmax_norm(v, mn, mx) for v in vals], dtype=np.float32)
    return normed


## C) Model loader & encoding

In [14]:
from sentence_transformers import SentenceTransformer

_MODEL_CACHE = {}

def get_model(name_model: str) -> SentenceTransformer:
    if name_model not in _MODEL_CACHE:
        _MODEL_CACHE[name_model] = SentenceTransformer(name_model)
    return _MODEL_CACHE[name_model]

@torch.no_grad()
def encode_texts(texts, name_model: str, device=None) -> torch.Tensor:
    m = get_model(name_model)
    embs = m.encode(texts, convert_to_tensor=True, device=device)
    # normalizamos a norma 1 para usar producto como coseno
    embs = torch.nn.functional.normalize(embs, p=2, dim=1)
    return embs


## D) Búsqueda de vecinos con normalización local + filtros

In [15]:
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd
import torch

def _ensure_l2_normalized(emb: np.ndarray) -> np.ndarray:
    """Asegura normalización L2 fila a fila para que dot == coseno."""
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12
    return emb / norms

def _cosine_matrix(emb: np.ndarray) -> np.ndarray:
    """Matriz de similitud coseno (asume emb L2 normalizado)."""
    return emb @ emb.T

def _rowwise_zscore(sim_row: np.ndarray) -> np.ndarray:
    """
    Z-score por fila: (x - mu) / sigma, con protección a sigma≈0,
    calculado SOLO sobre entradas finitas. Las no finitas (p.ej., -inf de la diagonal)
    se devuelven como -inf para que no pasen umbrales.
    """
    out = np.full_like(sim_row, -np.inf, dtype=float)
    mask = np.isfinite(sim_row)
    if not np.any(mask):
        return out
    vals = sim_row[mask]
    mu = float(vals.mean())
    sd = float(vals.std())
    if sd < 1e-12:
        out[mask] = 0.0
    else:
        out[mask] = (vals - mu) / (sd + 1e-12)
    return out

def _build_mnn_pairs(sim: np.ndarray, top_k: int) -> set:
    """
    Construye el conjunto de pares MNN:
    (i, j) está en MNN si i está en top_k de j y j está en top_k de i.
    Devuelve un set de tuplas (i, j) con i != j.
    """
    n = sim.shape[0]
    k_eff = min(top_k, max(1, n-1))
    # Top-k índices por fila (sin excluir self explícitamente; la diagonal es -inf)
    top_idx = np.argpartition(-sim, range(0, k_eff), axis=1)[:, :k_eff]
    # Ordenar por similitud real dentro del top parcial
    row_sorted = np.take_along_axis(sim, top_idx, axis=1)
    order = np.argsort(-row_sorted, axis=1)
    top_idx = np.take_along_axis(top_idx, order, axis=1)

    top_sets = [set(top_idx[i].tolist()) for i in range(n)]
    mnn = set()
    for i in range(n):
        for j in top_idx[i]:
            if i == j:
                continue
            if i in top_sets[j]:
                mnn.add((i, j))
    return mnn

def _mmr(
    query_vec: np.ndarray,
    cand_vecs: np.ndarray,
    cand_scores: np.ndarray,
    k_return: int,
    lambda_diversity: float = 0.0,
) -> List[int]:
    """
    Maximal Marginal Relevance (opcional). Si lambda_diversity = 0, devuelve
    los índices de cand_scores ordenados tal cual (relevancia pura).
    """
    m = cand_vecs.shape[0]
    if m == 0:
        return []
    if lambda_diversity <= 0.0:
        return np.argsort(-cand_scores)[:k_return].tolist()

    selected: List[int] = []
    candidates = set(range(m))
    cand_sim = cand_vecs @ cand_vecs.T  # coseno

    while candidates and len(selected) < k_return:
        best = None
        best_val = -1e18
        for c in list(candidates):
            rel = float(cand_scores[c])
            div = 0.0 if not selected else float(np.max(cand_sim[c, selected]))
            val = (1 - lambda_diversity) * rel - lambda_diversity * div
            if val > best_val:
                best_val = val
                best = c
        selected.append(best)
        candidates.remove(best)
    return selected

In [16]:
@torch.no_grad()
def nearest_neighbors_dwa(
    texts: List[str],
    labels: List[str],
    embeddings: torch.Tensor,
    *,
    k: int = 50,
    k_return: int = 10,
    tau_z: float = -np.inf,
    lam_len: float = 0.0,
    rho_stop: float = 0.0,
    use_mnn: bool = True,
    mnn_k: int = 50,
    lambda_diversity: float = 0.0,
    alpha_cos: float = 0.7,
    beta_bm25: float = 0.3,
    bm25_index = None,
) -> pd.DataFrame:
    """
    Recupera para cada DWA (query) sus k_return vecinos más relevantes
    dentro del propio catálogo, aplicando:
      - z-score por query (ignorando no finitos) y filtro por tau_z,
      - marca MNN (mutual nearest neighbor),
      - score final con penalizaciones suaves (longitud y stopwords),
      - (opcional) MMR para diversidad.
    """
    assert len(texts) == len(labels) == embeddings.shape[0], "Longitudes inconsistentes entre textos, labels y embeddings."
    N = embeddings.shape[0]

    # Asegurar numpy CPU
    if isinstance(embeddings, torch.Tensor):
        emb_np = embeddings.detach().cpu().numpy()
    else:
        emb_np = np.asarray(embeddings, dtype=np.float32)

    # L2-normalizar
    emb_np = _ensure_l2_normalized(emb_np)

    # Matriz coseno y exclusión del self-match
    sim = _cosine_matrix(emb_np)
    np.fill_diagonal(sim, -np.inf)

    # Grafo MNN (opcional)
    mnn_pairs = _build_mnn_pairs(sim, top_k=mnn_k) if use_mnn else set()

    rows = []
    max_k = min(k, N - 1)

    # Precalcular longitudes si las necesitas para otros análisis
    token_lens = [len(t.split()) for t in texts]

    for qi in range(N):
        q_text = texts[qi]
        q_label = labels[qi]

        # top-k por coseno (rápido) y ordenado
        row = sim[qi]
        if max_k > 0:
            idx_topk = np.argpartition(-row, range(0, max_k))[:max_k]
            idx_topk = idx_topk[np.argsort(-row[idx_topk])]
        else:
            idx_topk = np.array([], dtype=int)

        # z-score por query (ignorando no finitos)
        z_row = _rowwise_zscore(row)

        cand_info = []  # (j, sim, z, mnn, lenpen, stopjac, score)
        for j in idx_topk:
            z_ij = float(z_row[j])
            if z_ij < tau_z:
                continue

            # Hotfix: pasar TEXTO, no enteros, a length_penalty
            lenpen = float(length_penalty(q_text, texts[j]))   # <- aquí el fix
            stopjac = float(jaccard_stopwords(q_text, texts[j]))
            is_mnn = (qi, j) in mnn_pairs

            # --- NUEVO: BM25 normalizado por-candidatos (requiere bm25_index no nulo) ---
            if bm25_index is not None:
                # pasamos sólo los candidatos para normalizar min-max dentro de este vecindario
                # ojo: candidate_idx es idx_topk (array de ints)
                # reutilizamos los índices de este vecindario (normalización local)
                bm25_vec_local = bm25_query_scores(bm25_index, q_text, idx_topk)
                # necesitamos el valor correspondiente al j dentro del vector local
                # map: posición en idx_topk
                pos_j = int(np.where(idx_topk == j)[0][0]) if idx_topk.size else 0
                bm25j = float(bm25_vec_local[pos_j]) if idx_topk.size else 0.0
            else:
                bm25j = 0.0
            
            # --- Score híbrido ---
            s = alpha_cos * float(row[j]) \
                + beta_bm25 * bm25j \
                - lam_len * lenpen \
                - rho_stop * stopjac
            
            cand_info.append((j, float(row[j]), z_ij, bool(is_mnn), lenpen, stopjac, s))

        # ordenar por score descendente
        cand_info.sort(key=lambda t: t[6], reverse=True)

        # MMR opcional
        if lambda_diversity > 0.0 and len(cand_info) > 1:
            cand_idx = np.array([t[0] for t in cand_info], dtype=int)
            cand_vecs = emb_np[cand_idx]
            cand_scores = np.array([t[6] for t in cand_info], dtype=np.float32)
            selected_local = _mmr(
                query_vec=emb_np[qi],
                cand_vecs=cand_vecs,
                cand_scores=cand_scores,
                k_return=min(k_return, len(cand_info)),
                lambda_diversity=lambda_diversity,
            )
            cand_info = [cand_info[i] for i in selected_local]
        else:
            cand_info = cand_info[:min(k_return, len(cand_info))]

        # salida ancho
        out = {
            "query_idx": qi,
            "query_label": q_label,
            "query_text": q_text,
        }
        for pos in range(k_return):
            if pos < len(cand_info):
                j, sim_ij, z_ij, is_mnn, lenpen, stopjac, s = cand_info[pos]
                out[f"label_{pos+1}"]   = labels[j]
                out[f"sim_{pos+1}"]     = sim_ij
                out[f"z_{pos+1}"]       = z_ij
                out[f"mnn_{pos+1}"]     = int(is_mnn)
                out[f"lenpen_{pos+1}"]  = lenpen
                out[f"stopjac_{pos+1}"] = stopjac
                out[f"score_{pos+1}"]   = s
            else:
                out[f"label_{pos+1}"]   = None
                out[f"sim_{pos+1}"]     = np.nan
                out[f"z_{pos+1}"]       = np.nan
                out[f"mnn_{pos+1}"]     = np.nan
                out[f"lenpen_{pos+1}"]  = np.nan
                out[f"stopjac_{pos+1}"] = np.nan
                out[f"score_{pos+1}"]   = np.nan

        rows.append(out)

    df = pd.DataFrame(rows)
    return df

## E) Ejecutar para varios modelos y consolidar

In [17]:
import os
import re
import inspect
import numpy as np
import pandas as pd
import torch

# ------------------------------ Parámetros del experimento ---------------------------------------
CANDIDATE_MODELS = [
    "sentence-transformers/paraphrase-albert-small-v2",
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/distiluse-base-multilingual-cased-v2",
    "sentence-transformers/sentence-t5-base",
    "sentence-transformers/all-distilroberta-v1",
    "embedding-data/deberta-sentence-transformer",
    "sentence-transformers/paraphrase-MiniLM-L3-v2",
    "sentence-transformers/all-mpnet-base-v2"
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hiperparámetros coherentes con el Bloque D
K_NEIGHBORS   = 200#50        # vecindario bruto por coseno antes de penalizaciones
K_RETURN      = 50#10        # cuántos devolver tras scoring/MMR
TAU_Z         = 0.8#0.0, -np.inf   # umbral z-score por query (comienza sin filtro)
LAM_LEN       = 0.05      # penalización por desajuste de longitud (0–0.15 típico)
RHO_STOP      = 0.05      # penalización por solapamiento de stopwords (0–0.15 típico)
USE_MNN       = True      # marcar pares MNN (señal; no filtra por defecto)
MNN_K         = 50        # tamaño de top-k para grafo MNN
LAMBDA_DIV    = 0.0       # MMR desactivado en Sprint 1 (actívalo en Sprint 2)
BATCH_SIZE    = 64        # batch para encoding (fallback)

# Hiperparámetros Split 2
CLAUSE_POOLING = True      # activar pooling por cláusulas and/or
POOLING_MODE   = "mean"    # "mean" o "max" (mean suele ser estable)
LAMBDA_DIV     = 0.30      # activar MMR para diversidad (0.2–0.4 suele ir bien)

# Peso del score híbrido
ALPHA_COS      = 0.70
BETA_BM25      = 0.30



# Rutas
BASE         = os.getcwd()
DATA_DIR     = os.path.join(BASE, "data")
RESULTS_DIR  = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------ Compatibilidad con Bloque C --------------------------------------
# * Usa tu cache global si existe; si no, crea uno local.
try:
    _MODEL_CACHE  # noqa: F401
except NameError:
    _MODEL_CACHE = {}

In [18]:
def _get_model_cached(model_name: str, device: str = DEVICE):
    """Devuelve un SentenceTransformer cacheado."""
    from sentence_transformers import SentenceTransformer
    if model_name in _MODEL_CACHE:
        return _MODEL_CACHE[model_name]
    model = SentenceTransformer(model_name, device=device)
    _MODEL_CACHE[model_name] = model
    return model

def _encode_texts_fallback(model, texts, device: str = DEVICE, batch_size: int = BATCH_SIZE):
    """Fallback: devuelve embeddings en CPU como torch.Tensor (N, d)."""
    with torch.inference_mode():
        # Algunos SentenceTransformer ignoran el 'device' en encode si ya se inicializó con device
        emb = model.encode(
            texts,
            convert_to_numpy=True,
            batch_size=batch_size,
            show_progress_bar=False,
        )
    return torch.tensor(emb, dtype=torch.float32)

In [19]:
# ¿Existe encode_texts? (Bloque C)
try:
    encode_texts  # noqa: F401
    _HAS_ENCODE = True
except NameError:
    _HAS_ENCODE = False

In [20]:
def _get_embeddings_for_model(model_name: str, texts: list, device: str = DEVICE):
    """
    Obtiene embeddings usando:
      - TU encode_texts si existe (detectando firma y probando rutas típicas),
      - o fallback seguro _encode_texts_fallback.
    Devuelve torch.Tensor en CPU.
    """
    model = _get_model_cached(model_name, device=device)

    if not _HAS_ENCODE:
        return _encode_texts_fallback(model, texts, device=device, batch_size=BATCH_SIZE)

    # Detectar la firma de encode_texts y probar rutas de llamada compatibles
    sig = inspect.signature(encode_texts)
    param_names = list(sig.parameters.keys())

    # Rutas más comunes en tu código histórico:
    #  1) encode_texts(texts, name_model, device=None)
    #  2) encode_texts(model, texts, device=None)
    #  3) encode_texts(texts, model, device=None)  # menos común, pero lo soportamos
    tried = []

    # Ruta 1: (texts, name_model)
    try:
        if len(param_names) >= 2 and param_names[0] in ("texts", "text_list") and param_names[1] in ("name_model", "model_name"):
            emb = encode_texts(texts, model_name, device=device)
            if isinstance(emb, torch.Tensor):
                return emb.detach().cpu()
            return torch.tensor(np.asarray(emb), dtype=torch.float32)
        tried.append("encode_texts(texts, model_name)")
    except Exception as e:
        tried.append(f"FAILED: encode_texts(texts, model_name) -> {e}")

    # Ruta 2: (model, texts)
    try:
        emb = encode_texts(model, texts, device=device)
        if isinstance(emb, torch.Tensor):
            return emb.detach().cpu()
        return torch.tensor(np.asarray(emb), dtype=torch.float32)
    except Exception as e:
        tried.append(f"FAILED: encode_texts(model, texts) -> {e}")

    # Ruta 3: (texts, model)
    try:
        emb = encode_texts(texts, model, device=device)
        if isinstance(emb, torch.Tensor):
            return emb.detach().cpu()
        return torch.tensor(np.asarray(emb), dtype=torch.float32)
    except Exception as e:
        tried.append(f"FAILED: encode_texts(texts, model) -> {e}")

    # Si todas fallan, usa fallback y deja constancia en consola
    print("[E][WARN] No se pudo usar tu encode_texts con ninguna firma conocida. Intentos:", tried)
    return _encode_texts_fallback(model, texts, device=device, batch_size=BATCH_SIZE)

In [21]:
def compute_clause_pooled_embeddings(model_name: str, texts: list, device: str = DEVICE, mode: str = "mean"):
    """
    Reemplaza cada DWA por la media (o máx) de los embeddings de sus cláusulas (split and/or).
    No toca el corpus: devuelve un array (N, d) del mismo tamaño.
    """
    model = _get_model_cached(model_name, device=device)

    # Codificamos todas las cláusulas en un solo batch por eficiencia
    clauses_per_text = [split_clauses(t) for t in texts]
    all_clauses = []
    map_offsets = []  # (start, end) por texto
    start = 0
    for cl in clauses_per_text:
        if not cl:
            cl = [normalize_text(str(texts[len(map_offsets)]))]  # fallback
        all_clauses.extend(cl)
        end = start + len(cl)
        map_offsets.append((start, end))
        start = end

    if not all_clauses:
        # fallback: usar embeddings oracionales
        return _get_embeddings_for_model(model_name, texts, device=device)

    # Embeddings de todas las cláusulas
    clause_emb = _get_embeddings_for_model(model_name, all_clauses, device=device)
    if isinstance(clause_emb, torch.Tensor):
        clause_emb = clause_emb.detach().cpu().numpy()
    else:
        clause_emb = np.asarray(clause_emb)

    # Pooling por texto
    pooled = []
    for (s, e) in map_offsets:
        seg = clause_emb[s:e]
        if mode == "max":
            pooled.append(seg.max(axis=0))
        else:
            pooled.append(seg.mean(axis=0))
    return torch.tensor(np.vstack(pooled), dtype=torch.float32)


In [22]:
CORPUS_TEXTS  = onetLabels
CORPUS_LABELS = CORPUS_TEXTS[:]  # etiqueta = título en este dataset

In [23]:
# ------------------------------ Helper: ancho -> largo -------------------------------------------
def _wide_to_long_neighbors(df_wide: pd.DataFrame, model_name: str, k_return: int = K_RETURN) -> pd.DataFrame:
    """
    Convierte la salida ANCHA de nearest_neighbors_dwa a formato LARGO,
    conservando señales: sim, z, mnn, lenpen, stopjac, score.
    """
    rows = []
    for _, row in df_wide.iterrows():
        base = {
            "model": model_name,
            "query_idx": int(row["query_idx"]),
            "query_label": row["query_label"],
            "query_text": row["query_text"],
        }
        for r in range(1, k_return + 1):
            label = row.get(f"label_{r}", None)
            if label is None or (isinstance(label, float) and pd.isna(label)):
                continue
            rows.append({
                **base,
                "rank": r,
                "cand_label": label,
                "sim":     row.get(f"sim_{r}",     np.nan),
                "z":       row.get(f"z_{r}",       np.nan),
                "mnn":     row.get(f"mnn_{r}",     np.nan),
                "lenpen":  row.get(f"lenpen_{r}",  np.nan),
                "stopjac": row.get(f"stopjac_{r}", np.nan),
                "score":   row.get(f"score_{r}",   np.nan),
            })
    return pd.DataFrame(rows)

def _sanitize_for_filename(name: str) -> str:
    """Convierte el nombre de modelo en un string seguro para nombres de archivo."""
    s = re.sub(r"[^A-Za-z0-9_.-]+", "__", name)
    return s.strip("._-")

In [24]:
# Índice BM25 (léxico) sobre el corpus
BM25 = build_bm25_index(CORPUS_TEXTS)


In [25]:
# ------------------------------ Ejecución multi-modelo -------------------------------------------
neighbors_long_list = []
model_summaries     = []

# Verificación preventiva: Bloque D debe existir
assert "nearest_neighbors_dwa" in globals(), "nearest_neighbors_dwa no está definido. Ejecuta el Bloque D antes."

for model_name in CANDIDATE_MODELS:
    print(f"[E] Procesando modelo: {model_name}")

    # 1) Embeddings (oracionales o por cláusulas)
    if CLAUSE_POOLING:
        emb = compute_clause_pooled_embeddings(model_name, CORPUS_TEXTS, device=DEVICE, mode=POOLING_MODE)
    else:
        emb = _get_embeddings_for_model(model_name, CORPUS_TEXTS, device=DEVICE)
    
    if isinstance(emb, torch.Tensor) and emb.device.type != "cpu":
        emb = emb.detach().cpu()
    
    # 2) Recuperación con score híbrido y MMR activado
    df_neighbors = nearest_neighbors_dwa(
        texts=CORPUS_TEXTS,
        labels=CORPUS_LABELS,
        embeddings=emb,
        k=K_NEIGHBORS,
        k_return=K_RETURN,
        tau_z=TAU_Z,
        lam_len=LAM_LEN,
        rho_stop=RHO_STOP,
        use_mnn=USE_MNN,
        mnn_k=MNN_K,
        lambda_diversity=LAMBDA_DIV,   # ← activamos MMR
        alpha_cos=ALPHA_COS,           # ← pesos híbridos
        beta_bm25=BETA_BM25,
        bm25_index=BM25,               # ← índice léxico
    )


    # 3) Guardado por modelo (ANCHO)
    safe_model = _sanitize_for_filename(model_name)
    out_path_wide = os.path.join(RESULTS_DIR, f"neighbors_{safe_model}.csv")
    df_neighbors.to_csv(out_path_wide, index=False, encoding="utf-8")
    print(f"     → Guardado formato ancho: {out_path_wide} | filas={len(df_neighbors)}")

    # 4) Convertir a LARGO para consolidado y evaluación posterior
    df_long = _wide_to_long_neighbors(df_neighbors, model_name=model_name, k_return=K_RETURN)
    neighbors_long_list.append(df_long)

    # 5) Resumen por modelo (inspección rápida)
    if not df_long.empty:
        candidates_per_q = df_long.groupby("query_idx").size()
        avg_cand = float(candidates_per_q.mean())
        coverage = float((candidates_per_q > 0).mean())
    else:
        avg_cand, coverage = 0.0, 0.0

    model_summaries.append({
        "model": model_name,
        "num_queries": len(df_neighbors),
        "avg_candidates_per_query": avg_cand,
        "coverage_q_with_any": coverage,
        "saved_csv_ancho": out_path_wide,
    })

# 6) Consolidado global (LARGO)
if neighbors_long_list:
    df_neighbors_all_long = pd.concat(neighbors_long_list, ignore_index=True)
else:
    df_neighbors_all_long = pd.DataFrame(columns=[
        "model", "query_idx", "query_label", "query_text", "rank",
        "cand_label", "sim", "z", "mnn", "lenpen", "stopjac", "score"
    ])

out_path_long = os.path.join(RESULTS_DIR, "neighbors_all_long.csv")
df_neighbors_all_long.to_csv(out_path_long, index=False, encoding="utf-8")

# 7) Resumen de modelos
df_model_summary = pd.DataFrame(model_summaries)
out_path_summary = os.path.join(RESULTS_DIR, "neighbors_models_summary.csv")
df_model_summary.to_csv(out_path_summary, index=False, encoding="utf-8")

print(f"[E] Consolidado largo guardado: {out_path_long} | filas={len(df_neighbors_all_long)}")
print(f"[E] Resumen modelos guardado:  {out_path_summary}")

# Mostramos un vistazo si estás en notebook
try:
    display(df_model_summary)
except Exception:
    pass

[E] Procesando modelo: sentence-transformers/paraphrase-albert-small-v2
     → Guardado formato ancho: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/neighbors_sentence-transformers__paraphrase-albert-small-v2.csv | filas=221
[E] Procesando modelo: sentence-transformers/all-MiniLM-L6-v2
     → Guardado formato ancho: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/neighbors_sentence-transformers__all-MiniLM-L6-v2.csv | filas=221
[E] Procesando modelo: sentence-transformers/distiluse-base-multilingual-cased-v2
     → Guardado formato ancho: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/neighbors_sentence-transformers__distiluse-base-multilingual-cased-v2.csv | filas=221
[E] Procesando modelo: sentence-transformers/sentence-t5-base
     → Guardado formato ancho: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/neighbors_sentence-transformers__sentence-t5-base.csv | filas=221
[E] Procesando modelo: sentence-transformers/all-distilroberta-v1
     → Guardado formato ancho: /home/j

,model,num_queries,avg_candidates_per_query,coverage_q_with_any,saved_csv_ancho
0,sentence-transformers/paraphrase-albert-small-v2,221,43.886878,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
1,sentence-transformers/all-MiniLM-L6-v2,221,41.814480,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
2,sentence-transformers/distiluse-base-multiling...,221,40.918552,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
3,sentence-transformers/sentence-t5-base,221,42.986425,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
4,sentence-transformers/all-distilroberta-v1,221,41.416290,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
5,embedding-data/deberta-sentence-transformer,221,36.022624,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
6,sentence-transformers/paraphrase-MiniLM-L3-v2,221,42.176471,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
7,sentence-transformers/all-mpnet-base-v2,221,41.995475,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...


## F) Evaluación & Calibración 

In [26]:
import numpy as np
import pandas as pd
import torch
from typing import List, Dict, Tuple

# 1) Gold standard (puedes editarlo luego; empieza con algo manejable)
GOLD: Dict[str, Dict[str, List[str]]] = {
    "Train personnel on proper operational procedures.": {
        "positives": [
            "Conduct employee training programs.",
            "Instruct college students in physical or life sciences."
        ],
        "negatives": [
            "Prepare proposal documents.",
            "Purchase materials, equipment, or other resources."
        ]
    },
    "Prepare procedural documents.": {
        "positives": [
            "Document organizational or operational procedures.",
            "Prepare contracts, disclosures, or applications."
        ],
        "negatives": [
            "Operate industrial equipment.",
            "Recruit personnel."
        ]
    },
    "Design medical devices or appliances.": {
        "positives": [
            "Design electromechanical equipment or systems.",
            "Design industrial equipment."
        ],
        "negatives": [
            "Develop business or financial information systems.",
            "Supervise employees."
        ]
    },
    "Conduct environmental audits.": {
        "positives": [
            "Monitor activities affecting environmental quality.",
            "Analyze environmental regulations to ensure organizational compliance."
        ],
        "negatives": [
            "Direct sales, marketing, or customer service activities.",
            "Interview employees, customers, or others to collect information."
        ]
    },
    "Develop operational methods or processes that use green materials or emphasize sustainability.": {
        "positives": [
            "Develop sustainable business strategies or practices.",
            "Implement transportation changes to reduce environmental impact."
        ],
        "negatives": [
            "Manage human resources activities.",
            "Fabricate devices or components."
        ]
    },
    "Estimate operational costs.": {
        "positives": [
            "Estimate labor requirements.",
            "Estimate time requirements for development or production projects."
        ],
        "negatives": [
            "Research genetic characteristics or expression.",
            "Inspect finished products to locate flaws."
        ]
    },
    "Supervise engineering or other technical personnel.": {
        "positives": [
            "Supervise production or support personnel.",
            "Supervise scientific or technical personnel."
        ],
        "negatives": [
            "Prepare financial documents.",
            "Analyze chemical compounds or substances."
        ]
    },
    "Research engineering aspects of biological or chemical processes.": {
        "positives": [
            "Research microbiological or chemical processes or structures.",
            "Research engineering applications of emerging technologies."
        ],
        "negatives": [
            "Manage inventories of products or organizational resources.",
            "Negotiate labor disputes."
        ]
    },
    "Develop software or computer applications.": {
        "positives": [
            "Develop business or financial information systems.",
            "Program robotic equipment."
        ],
        "negatives": [
            "Hire personnel.",
            "Administer standardized physical or psychological tests."
        ]
    },
    "Evaluate quality of materials or products.": {
        "positives": [
            "Test quality of materials or finished products.",
            "Inspect finished products to locate flaws."
        ],
        "negatives": [
            "Direct organizational operations, projects, or services.",
            "Perform marketing activities."
        ]
    }
}


In [27]:
import os
import json
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------------
# Parámetros de evaluación
# ------------------------------------------------------------------------------------
RESULTS_DIR     = os.path.join(os.getcwd(), "results")
NEIGHBORS_LONG  = os.path.join(RESULTS_DIR, "neighbors_all_long.csv")

# Campo base para umbral (puede ser "score" o "sim")
SCORE_FIELD     = "score"      # recomendado: "score" (ya incluye penalizaciones)
THRESHOLDS      = np.round(np.linspace(-0.2, 0.9, 56), 3)  # barrido amplio (ajústalo a tu rango real)
K_LIST          = [1, 5, 10]   # K para métricas @K

# Si deseas evaluar "aceptación" con top-1 únicamente (recomendado) deja True
DECISION_TOP1   = True

# ------------------------------------------------------------------------------------
# Carga del consolidado del Bloque E
# Espera columnas: model, query_idx, query_label, query_text, rank, cand_label, sim, z, mnn, lenpen, stopjac, score
# ------------------------------------------------------------------------------------
assert os.path.isfile(NEIGHBORS_LONG), f"No encuentro {NEIGHBORS_LONG}. Ejecuta el Bloque E antes."
dfL = pd.read_csv(NEIGHBORS_LONG)

# Validación ligera
required_cols = {"model","query_label","query_text","rank","cand_label","sim","z","score"}
missing = required_cols - set(dfL.columns)
assert not missing, f"Faltan columnas en neighbors_all_long.csv: {missing}"

In [28]:
# ------------------------------------------------------------------------------------
# GOLD: Adaptadores (usa cualquiera de estos 3 enfoques)
#  1) GOLD_DICT: dict {"query_label": {"positives":[...], "negatives":[...]}}
#  2) GOLD_DF: DataFrame con columnas: query_label, positives (json/list), negatives (opcional)
#  3) GOLD_JSON_PATH: ruta a un json con estructura del caso (1)
# IMPORTANTE: Los labels de GOLD deben existir en onetLabels['dwa_title'] y en dfL['query_label'].
# ------------------------------------------------------------------------------------

# === EJEMPLO de un GOLD embebido (edita/ sustituye por el tuyo) ===
# Si ya tienes tu GOLD cargado en otra celda (p.ej. GOLD_DICT), comenta esta sección.
#GOLD_DICT = {
    # "query_label": { "positives": [lista de labels válidos], "negatives": [opcional] }
    #"Calibrate quality control instruments": {"positives": ["Test quality control instruments", "Maintain quality records"]},
    #"Assemble production components": {"positives": ["Inspect assembled components", "Test assembled components"]},
    # ... completa con tu mini-GOLD real ...
#}

# Si tu GOLD ya existe con otro nombre/forma, adapta con los helpers de abajo:
GOLD_JSON_PATH = None  # p.ej., os.path.join("data","gold_dwas.json")
GOLD_DF = None         # p.ej., DataFrame con query_label, positives, negatives



In [29]:
def _gold_from_dict(gd: dict) -> dict:
    """Valida/normaliza GOLD de tipo dict."""
    out = {}
    for q, v in gd.items():
        poss = v.get("positives", [])
        negs = v.get("negatives", [])
        # Aseguramos lista
        if isinstance(poss, (str,)):
            poss = [poss]
        if isinstance(negs, (str,)):
            negs = [negs]
        out[q] = {"positives": list(map(str, poss)), "negatives": list(map(str, negs))}
    return out

def _gold_from_df(gdf: pd.DataFrame) -> dict:
    """Convierte un DataFrame GOLD a dict estándar."""
    assert "query_label" in gdf.columns, "GOLD_DF debe tener 'query_label'."
    out = {}
    for _, r in gdf.iterrows():
        q = str(r["query_label"])
        poss = r.get("positives", [])
        negs = r.get("negatives", [])
        # intentar json.loads si es string
        if isinstance(poss, str):
            try:
                poss = json.loads(poss)
            except Exception:
                poss = [poss]
        if isinstance(negs, str) and len(str(negs).strip()) > 0:
            try:
                negs = json.loads(negs)
            except Exception:
                negs = [negs]
        out[q] = {"positives": [str(x) for x in (poss or [])], "negatives": [str(x) for x in (negs or [])]}
    return out

def _gold_from_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        gd = json.load(f)
    return _gold_from_dict(gd)

In [30]:
# Prioridad de carga: JSON > DF > dict embebido
#if GOLD_JSON_PATH:
#    GOLD = _gold_from_json(GOLD_JSON_PATH)
#elif GOLD_DF is not None:
#    GOLD = _gold_from_df(GOLD_DF)
#else:
#    GOLD = _gold_from_dict(GOLD_DICT)

# Limpiar GOLD a las queries que existen en dfL
valid_queries = set(dfL["query_label"].astype(str).unique().tolist())
GOLD = {q: v for q, v in GOLD.items() if q in valid_queries}
assert len(GOLD) > 0, "GOLD quedó vacío tras cruzar con dfL. Verifica etiquetas."

In [31]:
# ------------------------------------------------------------------------------------
# Helpers de relevancia y métricas
# ------------------------------------------------------------------------------------
def _relevance_vector_for_query(df_q: pd.DataFrame, positives: set) -> np.ndarray:
    """
    Dado el sub-DataFrame de una query (ordenado por rank ascendente),
    devuelve un vector binario de relevancias [1 si cand_label in positives, 0 en caso contrario].
    """
    return df_q["cand_label"].astype(str).isin(positives).astype(int).values

def _dcg_at_k(rels: np.ndarray, k: int) -> float:
    rels = rels[:k]
    if rels.size == 0:
        return 0.0
    # DCG: sum((2^rel - 1) / log2(i+2))
    idx = np.arange(1, len(rels) + 1)
    return float(np.sum((2**rels - 1) / np.log2(idx + 1)))

def _ndcg_at_k(rels: np.ndarray, k: int) -> float:
    dcg = _dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = _dcg_at_k(ideal, k)
    return float(dcg / idcg) if idcg > 0 else 0.0

def _precision_at_k(rels: np.ndarray, k: int) -> float:
    rels = rels[:k]
    return float(rels.mean()) if rels.size > 0 else 0.0

def _recall_at_k(rels: np.ndarray, k: int, total_pos: int) -> float:
    if total_pos <= 0:
        return 0.0
    return float(rels[:k].sum() / total_pos)

def _mrr(rels: np.ndarray) -> float:
    # Reciprocal rank del primer relevante
    rr = 0.0
    for i, r in enumerate(rels, start=1):
        if r > 0:
            rr = 1.0 / i
            break
    return float(rr)

# ------------------------------------------------------------------------------------
# Evaluación por modelo: ranking @K
# ------------------------------------------------------------------------------------
def evaluate_ranking_by_model(df_long: pd.DataFrame, gold: dict, k_list=K_LIST):
    """
    Calcula nDCG@K, Recall@K, Precision@K y MRR por query y promedia.
    Devuelve: (per_query_df, summary_df)
    """
    records = []
    for model, dfM in df_long.groupby("model"):
        for q_label, spec in gold.items():
            dfQ = dfM[dfM["query_label"] == q_label].sort_values("rank", ascending=True)
            if dfQ.empty:
                continue
            positives = set(spec.get("positives", []))
            rels = _relevance_vector_for_query(dfQ, positives)

            metrics = {"model": model, "query_label": q_label}
            # MRR sobre ranking completo
            metrics["MRR"] = _mrr(rels)
            # @K
            total_pos = max(1, len(positives))
            for K in k_list:
                metrics[f"nDCG@{K}"] = _ndcg_at_k(rels, K)
                metrics[f"Recall@{K}"] = _recall_at_k(rels, K, total_pos)
                metrics[f"Precision@{K}"] = _precision_at_k(rels, K)
            records.append(metrics)

    per_query = pd.DataFrame(records)
    if per_query.empty:
        raise AssertionError("No hay métricas calculadas. ¿dfL vacío o GOLD no hace match?")

    # Resumen por modelo
    agg = { "MRR": "mean" }
    for K in k_list:
        agg[f"nDCG@{K}"] = "mean"
        agg[f"Recall@{K}"] = "mean"
        agg[f"Precision@{K}"] = "mean"
    summary = per_query.groupby("model", as_index=False).agg(agg)

    return per_query, summary

# ------------------------------------------------------------------------------------
# Barrido de umbrales (aceptación binaria)
# Regla por defecto: decisión en Top-1. Si top1[SCORE_FIELD] >= thr -> pred = cand_label_top1; si no, abstención.
# Métricas: precision, recall, f1, coverage (porcentaje de queries con predicción emitida).
# ------------------------------------------------------------------------------------
def threshold_sweep_by_model(df_long: pd.DataFrame, gold: dict, thresholds=THRESHOLDS, score_field=SCORE_FIELD, decision_top1=DECISION_TOP1):
    rows = []
    for model, dfM in df_long.groupby("model"):
        for thr in thresholds:
            n_pos, n_pred, n_truepos = 0, 0, 0
            for q_label, spec in gold.items():
                positives = set(spec.get("positives", []))
                n_pos += 1  # contamos una "query positiva" si existe al menos un positivo definido
                dfQ = dfM[dfM["query_label"] == q_label].sort_values("rank", ascending=True)
                if dfQ.empty:
                    continue

                # Decisión
                if decision_top1:
                    top1 = dfQ.iloc[0]
                    sc = float(top1.get(score_field, np.nan))
                    if math.isnan(sc) or sc < thr:
                        # abstención
                        continue
                    # predicción = label del top1
                    n_pred += 1
                    if str(top1["cand_label"]) in positives:
                        n_truepos += 1
                else:
                    # Alternativa: aceptar si "existe" algún candidato >= thr; pred = mejor de entre ellos
                    dfA = dfQ[dfQ[score_field] >= thr]
                    if dfA.empty:
                        continue
                    n_pred += 1
                    # si el mejor entre aceptados es positivo, cuenta como acierto
                    cand_best = dfA.sort_values(score_field, ascending=False).iloc[0]
                    if str(cand_best["cand_label"]) in positives:
                        n_truepos += 1

            precision = (n_truepos / n_pred) if n_pred > 0 else 0.0
            recall    = (n_truepos / n_pos)  if n_pos  > 0 else 0.0
            f1        = (2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0.0
            coverage  = (n_pred / n_pos) if n_pos > 0 else 0.0

            rows.append({
                "model": model,
                "threshold": float(thr),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "coverage": coverage,
                "n_queries": n_pos,
                "n_predictions": n_pred,
                "n_truepos": n_truepos,
                "score_field": score_field,
                "decision_top1": bool(decision_top1),
            })
    df_thr = pd.DataFrame(rows)
    # Elegimos por modelo el mejor umbral según F1 (desempate: mayor coverage)
    best_rows = []
    for model, dfM in df_thr.groupby("model"):
        if dfM.empty:
            continue
        df_sorted = dfM.sort_values(["f1","coverage"], ascending=[False, False])
        best_rows.append({**df_sorted.iloc[0].to_dict(), "model": model})
    df_best = pd.DataFrame(best_rows)
    return df_thr, df_best

In [32]:
# ------------------------------------------------------------------------------------
# Ejecutar evaluación
# ------------------------------------------------------------------------------------
print("[F] Evaluando métricas de ranking por modelo...")
DF_PER_QUERY, RANKING_SUMMARY = evaluate_ranking_by_model(dfL, GOLD, k_list=K_LIST)

print("[F] Barrido de umbrales...")
DF_THR, BEST_THR = threshold_sweep_by_model(
    dfL, GOLD, thresholds=THRESHOLDS, score_field=SCORE_FIELD, decision_top1=DECISION_TOP1
)
# Guardado de salidas principales
out_per_query = os.path.join(RESULTS_DIR, "eval_per_query.csv")
out_summary   = os.path.join(RESULTS_DIR, "eval_ranking_summary.csv")
out_thr       = os.path.join(RESULTS_DIR, "eval_thresholds.csv")
out_best      = os.path.join(RESULTS_DIR, "eval_best_thresholds.csv")

DF_PER_QUERY.to_csv(out_per_query, index=False, encoding="utf-8")
RANKING_SUMMARY.to_csv(out_summary, index=False, encoding="utf-8")
DF_THR.to_csv(out_thr, index=False, encoding="utf-8")
BEST_THR.to_csv(out_best, index=False, encoding="utf-8")

print(f"[F] Guardado per-query:        {out_per_query} (filas={len(DF_PER_QUERY)})")
print(f"[F] Guardado ranking summary:  {out_summary}")
print(f"[F] Guardado thresholds sweep: {out_thr} (filas={len(DF_THR)})")
print(f"[F] Guardado best thresholds:  {out_best}")

display(RANKING_SUMMARY.sort_values(["nDCG@10","Recall@1"], ascending=[False, False]))
display(BEST_THR.sort_values("f1", ascending=False))

[F] Evaluando métricas de ranking por modelo...
[F] Barrido de umbrales...
[F] Guardado per-query:        /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/eval_per_query.csv (filas=80)
[F] Guardado ranking summary:  /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/eval_ranking_summary.csv
[F] Guardado thresholds sweep: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/eval_thresholds.csv (filas=448)
[F] Guardado best thresholds:  /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/eval_best_thresholds.csv


,model,MRR,nDCG@1,Recall@1,Precision@1,nDCG@5,Recall@5,Precision@5,nDCG@10,Recall@10,Precision@10
4,sentence-transformers/distiluse-base-multiling...,0.465202,0.3,0.15,0.3,0.416914,0.45,0.18,0.509734,0.65,0.13
3,sentence-transformers/all-mpnet-base-v2,0.415278,0.2,0.10,0.2,0.360842,0.45,0.18,0.462761,0.70,0.14
1,sentence-transformers/all-MiniLM-L6-v2,0.401458,0.2,0.10,0.2,0.358155,0.45,0.18,0.440732,0.65,0.13
7,sentence-transformers/sentence-t5-base,0.405867,0.2,0.10,0.2,0.353904,0.45,0.18,0.432364,0.65,0.13
0,embedding-data/deberta-sentence-transformer,0.385498,0.3,0.15,0.3,0.354377,0.30,0.12,0.423331,0.40,0.08
6,sentence-transformers/paraphrase-albert-small-v2,0.380078,0.2,0.10,0.2,0.396840,0.45,0.18,0.415297,0.50,0.10
2,sentence-transformers/all-distilroberta-v1,0.357450,0.1,0.05,0.1,0.348530,0.50,0.20,0.411247,0.65,0.13
5,sentence-transformers/paraphrase-MiniLM-L3-v2,0.315972,0.1,0.05,0.1,0.273846,0.40,0.16,0.367533,0.65,0.13


,model,threshold,precision,recall,f1,coverage,n_queries,n_predictions,n_truepos,score_field,decision_top1
0,embedding-data/deberta-sentence-transformer,0.80,0.333333,0.3,0.315789,0.9,10,9,3,score,True
4,sentence-transformers/distiluse-base-multiling...,0.56,0.333333,0.3,0.315789,0.9,10,9,3,score,True
3,sentence-transformers/all-mpnet-base-v2,0.68,0.500000,0.2,0.285714,0.4,10,4,2,score,True
7,sentence-transformers/sentence-t5-base,0.88,0.333333,0.2,0.250000,0.6,10,6,2,score,True
1,sentence-transformers/all-MiniLM-L6-v2,0.58,0.222222,0.2,0.210526,0.9,10,9,2,score,True
6,sentence-transformers/paraphrase-albert-small-v2,-0.20,0.200000,0.2,0.200000,1.0,10,10,2,score,True
2,sentence-transformers/all-distilroberta-v1,0.80,1.000000,0.1,0.181818,0.1,10,1,1,score,True
5,sentence-transformers/paraphrase-MiniLM-L3-v2,0.76,0.333333,0.1,0.153846,0.3,10,3,1,score,True


In [33]:
# ------------------------------------------------------------------------------------
# Selección de modelo ganador + extracción de FP/FN alrededor del umbral
# ------------------------------------------------------------------------------------
def select_winner_model(ranking_summary: pd.DataFrame) -> str:
    """
    Elige modelo ganador por nDCG@10 (empate: Recall@1).
    """
    tmp = ranking_summary.copy()
    tmp = tmp.sort_values(["nDCG@10", "Recall@1"], ascending=[False, False])
    return str(tmp.iloc[0]["model"])


In [34]:
WINNER = select_winner_model(RANKING_SUMMARY)
print(f"[F] Modelo ganador (ranking): {WINNER}")

# Umbral recomendado para el ganador (por F1)
if not BEST_THR.empty and WINNER in BEST_THR["model"].values:
    RECOMMENDED_THR = float(BEST_THR[BEST_THR["model"] == WINNER].iloc[0]["threshold"])
else:
    RECOMMENDED_THR = float(np.median(THRESHOLDS))
print(f"[F] Umbral recomendado para {WINNER}: {RECOMMENDED_THR:.3f} (field={SCORE_FIELD})")


[F] Modelo ganador (ranking): sentence-transformers/distiluse-base-multilingual-cased-v2
[F] Umbral recomendado para sentence-transformers/distiluse-base-multilingual-cased-v2: 0.560 (field=score)


In [35]:
# ------------------------------------------------------------------------------------
# Construir tablas de FP/FN del ganador cerca del umbral
# Regla: predicción top1 con score>=thr -> positivo si cand_label ∈ GOLD. Si no, FP.
# Si existe positivo en top-10 pero score<thr -> FN (por "abstención excesiva").
# También computamos margen Δ = score_top1 - score_top2 y añadimos señales auxiliares.
# ------------------------------------------------------------------------------------
def false_pos_neg_tables(df_long: pd.DataFrame, gold: dict, model: str, threshold: float, score_field=SCORE_FIELD, topK=10):
    rows_fp, rows_fn = [], []
    dfM = df_long[df_long["model"] == model].copy()

    for q_label, spec in gold.items():
        dfQ = dfM[dfM["query_label"] == q_label].sort_values("rank", ascending=True)
        if dfQ.empty:
            continue
        positives = set(spec.get("positives", []))

        # Top-1 y margen
        top1 = dfQ.iloc[0]
        score1 = float(top1.get(score_field, np.nan))
        if len(dfQ) >= 2:
            top2 = dfQ.iloc[1]
            score2 = float(top2.get(score_field, np.nan))
            delta = score1 - score2
        else:
            delta = np.nan

        is_pos_top1 = str(top1["cand_label"]) in positives

        # FP: emitimos predicción (score1>=thr) pero no es positivo
        if not math.isnan(score1) and score1 >= threshold and not is_pos_top1:
            rows_fp.append({
                "query_label": q_label,
                "top1_label": str(top1["cand_label"]),
                "top1_score": score1,
                "top1_sim": float(top1.get("sim", np.nan)),
                "top1_z": float(top1.get("z", np.nan)),
                "delta_top1_top2": float(delta),
                "top1_lenpen": float(top1.get("lenpen", np.nan)),
                "top1_stopjac": float(top1.get("stopjac", np.nan)),
                "top1_mnn": int(top1.get("mnn", 0)) if not pd.isna(top1.get("mnn", np.nan)) else 0,
                "positives": list(positives),
            })

        # FN: hay al menos un positivo dentro del topK, pero score1<thr → abstención (perdimos un caso positivo)
        if not math.isnan(score1) and score1 < threshold:
            dfTopK = dfQ.head(topK)
            any_pos_in_topK = dfTopK["cand_label"].astype(str).isin(positives).any()
            if any_pos_in_topK:
                # Trae el positivo mejor rankeado
                pos_row = dfTopK[dfTopK["cand_label"].astype(str).isin(positives)].iloc[0]
                rows_fn.append({
                    "query_label": q_label,
                    "bestpos_label": str(pos_row["cand_label"]),
                    "bestpos_rank": int(pos_row["rank"]),
                    "bestpos_score": float(pos_row.get(score_field, np.nan)),
                    "bestpos_sim": float(pos_row.get("sim", np.nan)),
                    "bestpos_z": float(pos_row.get("z", np.nan)),
                    "top1_label": str(top1["cand_label"]),
                    "top1_score": score1,
                    "top1_sim": float(top1.get("sim", np.nan)),
                    "top1_z": float(top1.get("z", np.nan)),
                    "delta_top1_top2": float(delta),
                    "top1_lenpen": float(top1.get("lenpen", np.nan)),
                    "top1_stopjac": float(top1.get("stopjac", np.nan)),
                    "top1_mnn": int(top1.get("mnn", 0)) if not pd.isna(top1.get("mnn", np.nan)) else 0,
                })

    return pd.DataFrame(rows_fp), pd.DataFrame(rows_fn)

In [36]:
# Tablas de FP/FN del ganador cerca del umbral
DF_FP, DF_FN = false_pos_neg_tables(
    dfL, GOLD, WINNER, RECOMMENDED_THR,
    score_field=SCORE_FIELD, topK=10
)

# Guardar para revisión manual / error analysis (usando nombre de archivo saneado)
safe_winner = _sanitize_for_filename(WINNER)

out_fp = os.path.join(RESULTS_DIR, f"errors_fp_{safe_winner}.csv")
out_fn = os.path.join(RESULTS_DIR, f"errors_fn_{safe_winner}.csv")

# Asegura que la carpeta de resultados existe (por si se cambió RESULTS_DIR arriba)
os.makedirs(RESULTS_DIR, exist_ok=True)

DF_FP.to_csv(out_fp, index=False, encoding="utf-8")
DF_FN.to_csv(out_fn, index=False, encoding="utf-8")

print(f"[F] FP guardados en: {out_fp} (filas={len(DF_FP)})")
print(f"[F] FN guardados en: {out_fn} (filas={len(DF_FN)})")

[F] FP guardados en: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/errors_fp_sentence-transformers__distiluse-base-multilingual-cased-v2.csv (filas=6)
[F] FN guardados en: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/errors_fn_sentence-transformers__distiluse-base-multilingual-cased-v2.csv (filas=0)


In [37]:
# --- F) Calibración robusta (isotónica -> Platt -> rank) con CV (Split 2 hotfix) -----------------
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from collections import defaultdict
import numpy as np
import pandas as pd
import os

USE_CALIBRATION = True
CAL_TOPK = 3#1          # 1 = sólo top-1 (recomendado para "p(top1 es correcto)")
MIN_POS = 3           # mínimo positivos para isotónica
MIN_NEG = 3           # mínimo negativos para isotónica
N_FOLDS = 5           # K-fold por queries (reduce overfitting); usa 5 o leave-one-out si GOLD pequeño
PROB_THRESHOLDS = np.round(np.linspace(0.3, 0.9, 13), 2)
CAL_SCORE_FIELD = SCORE_FIELD  # normalmente "score"

In [38]:
def _build_calibration_df(df_long: pd.DataFrame, gold: dict, model: str, topk=1, score_field="score"):
    """
    Construye dataset de calibración para un modelo:
      X: score del candidato rank<=topk (usamos rank=1 por defecto)
      y: 1 si cand_label ∈ positives(query), 0 si no
    Retorna DataFrame con columnas: query_label, score, y
    """
    sub = df_long[df_long["model"] == model].copy()
    rows = []
    for q, spec in gold.items():
        pos = set(spec.get("positives", []))
        dq = sub[sub["query_label"] == q].sort_values("rank", ascending=True).head(topk)
        if dq.empty: 
            continue
        # para topk>1, tomamos el mejor score entre los topK (y=1 si cualquiera es positivo)
        cand_best = dq.iloc[0]
        s = float(cand_best.get(score_field, np.nan))
        y = 1 if str(cand_best["cand_label"]) in pos else 0
        if topk > 1 and len(dq) > 1:
            # si cualquiera de los topK es positivo, y=1; score = max score dentro de topK
            any_pos = dq["cand_label"].astype(str).isin(pos).any()
            y = 1 if any_pos else 0
            s = float(dq[score_field].max())
        if not np.isnan(s):
            rows.append({"query_label": q, "score": s, "y": y})
    return pd.DataFrame(rows)

class _Calibrator:
    """Wrapper para isotónica / platt / rank-based con interfaz predict_proba(scores)."""
    def __init__(self, kind, model=None, ref_scores=None, ref_probs=None):
        self.kind = kind              # "isotonic" | "platt" | "rank"
        self.model = model            # IsotonicRegression o LogisticRegression
        self.ref_scores = ref_scores  # para rank calibration (percentiles)
        self.ref_probs = ref_probs

    def predict_proba(self, scores: np.ndarray):
        scores = np.asarray(scores, dtype=float)
        if self.kind == "isotonic":
            return self.model.predict(scores)
        elif self.kind == "platt":
            # logistic returns prob for class 1
            return self.model.predict_proba(scores.reshape(-1,1))[:,1]
        else:  # rank
            # percentil rank -> [0,1]
            # mapea cada score a su CDF empírica
            ref = np.sort(self.ref_scores)
            ranks = np.searchsorted(ref, scores, side="right")
            probs = ranks / max(1, len(ref))
            return probs

def _fit_single_calibrator(df_cal: pd.DataFrame):
    """Elige isotónica si hay clases suficientes; si no, Platt; si no, rank-based."""
    if df_cal.empty:
        return None
    y = df_cal["y"].values
    x = df_cal["score"].values
    n_pos = int((y==1).sum())
    n_neg = int((y==0).sum())
    # Caso degenerado: todas las y iguales
    if len(set(y)) < 2:
        # rank-based sobre scores (mejor que nada)
        return _Calibrator(kind="rank", ref_scores=x)
    # Isotónica si hay suficientes
    if n_pos >= MIN_POS and n_neg >= MIN_NEG:
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(x, y)
        return _Calibrator(kind="isotonic", model=ir)
    # Fallback Platt (logistic)
    try:
        lr = LogisticRegression(class_weight="balanced", max_iter=1000)
        lr.fit(x.reshape(-1,1), y)
        return _Calibrator(kind="platt", model=lr)
    except Exception:
        return _Calibrator(kind="rank", ref_scores=x)

def _cv_calibrate_and_predict(df_long: pd.DataFrame, gold: dict, model: str, topk=1, score_field="score", n_folds=5):
    """
    Calibración CV por queries: divide queries del GOLD en K folds, entrena calibrador en K-1 y predice en el fold hold-out.
    Devuelve DataFrame con columnas: query_label, score, y, p
    """
    df_all = _build_calibration_df(df_long, gold, model, topk=topk, score_field=score_field)
    if df_all.empty:
        return df_all.assign(p=np.nan)
    # KFold por queries (ítems = queries)
    queries = df_all["query_label"].unique()
    if len(queries) < n_folds:
        n_folds = max(2, len(queries))  # lo que se pueda
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    preds = []
    for train_idx, test_idx in kf.split(queries):
        q_train = set(queries[train_idx])
        q_test  = set(queries[test_idx])
        df_train = df_all[df_all["query_label"].isin(q_train)]
        df_test  = df_all[df_all["query_label"].isin(q_test)].copy()
        cal = _fit_single_calibrator(df_train)
        if cal is None:
            df_test["p"] = np.nan
        else:
            df_test["p"] = cal.predict_proba(df_test["score"].values)
        preds.append(df_test)
    df_pred = pd.concat(preds, ignore_index=True)
    return df_pred

def threshold_sweep_calibrated_df(df_pred: pd.DataFrame, prob_thresholds=PROB_THRESHOLDS):
    """Barre umbrales sobre p y calcula precision/recall/f1/coverage."""
    out = []
    # Contamos n_pos como nº de queries con etiqueta en df_pred (una por query)
    # Si hay varias filas por query (topK>1), nos quedamos con la mejor (score más alto) ya se construyó así.
    df_q = df_pred.drop_duplicates(subset=["query_label"], keep="first")
    n_pos = len(df_q)
    for thr in prob_thresholds:
        # predicción = aceptar si p>=thr
        df_acc = df_q[df_q["p"] >= thr]
        n_pred = len(df_acc)
        n_truepos = int((df_acc["y"] == 1).sum())
        precision = (n_truepos / n_pred) if n_pred > 0 else 0.0
        recall    = (n_truepos / n_pos)  if n_pos  > 0 else 0.0
        f1        = (2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0.0
        coverage  = (n_pred / n_pos) if n_pos > 0 else 0.0
        out.append({"prob_threshold": thr, "precision": precision, "recall": recall, "f1": f1, "coverage": coverage,
                    "n_queries": n_pos, "n_predictions": n_pred, "n_truepos": n_truepos})
    return pd.DataFrame(out)

In [39]:
if USE_CALIBRATION:
    CAL_RESULTS = {}
    MODELS = DF_THR["model"].unique().tolist()
    for m in MODELS:
        df_pred = _cv_calibrate_and_predict(dfL, GOLD, m, topk=CAL_TOPK, score_field=CAL_SCORE_FIELD, n_folds=N_FOLDS)
        if df_pred.empty or df_pred["p"].isna().all():
            print(f"[F][CAL] No se pudo calibrar {m} (datos insuficientes).")
            continue
        df_sweep = threshold_sweep_calibrated_df(df_pred, prob_thresholds=PROB_THRESHOLDS)
        CAL_RESULTS[m] = {"pred": df_pred, "sweep": df_sweep}
        # Guardados
        safe = _sanitize_for_filename(m)
        df_pred.to_csv(os.path.join(RESULTS_DIR, f"cal_pred_{safe}.csv"), index=False, encoding="utf-8")
        df_sweep.to_csv(os.path.join(RESULTS_DIR, f"cal_sweep_{safe}.csv"), index=False, encoding="utf-8")
        print(f"[F][CAL] {m}: mejor F1={df_sweep['f1'].max():.3f} @ p≥{df_sweep.iloc[df_sweep['f1'].idxmax()]['prob_threshold']:.2f}")
    # Informe rápido del ganador si existe
    if WINNER in CAL_RESULTS:
        df_sweep = CAL_RESULTS[WINNER]["sweep"]
        print(f"[F][CAL] GANADOR {WINNER}: mejor F1={df_sweep['f1'].max():.3f} @ p≥{df_sweep.iloc[df_sweep['f1'].idxmax()]['prob_threshold']:.2f}")
        try:
            display(df_sweep.sort_values(["f1","coverage"], ascending=[False, False]).head(10))
        except Exception:
            pass

[F][CAL] embedding-data/deberta-sentence-transformer: mejor F1=0.353 @ p≥0.45
[F][CAL] sentence-transformers/all-MiniLM-L6-v2: mejor F1=0.235 @ p≥0.30
[F][CAL] sentence-transformers/all-distilroberta-v1: mejor F1=0.333 @ p≥0.30
[F][CAL] sentence-transformers/all-mpnet-base-v2: mejor F1=0.143 @ p≥0.45
[F][CAL] sentence-transformers/distiluse-base-multilingual-cased-v2: mejor F1=0.421 @ p≥0.30
[F][CAL] sentence-transformers/paraphrase-MiniLM-L3-v2: mejor F1=0.333 @ p≥0.30
[F][CAL] sentence-transformers/paraphrase-albert-small-v2: mejor F1=0.333 @ p≥0.55
[F][CAL] sentence-transformers/sentence-t5-base: mejor F1=0.500 @ p≥0.55
[F][CAL] GANADOR sentence-transformers/distiluse-base-multilingual-cased-v2: mejor F1=0.421 @ p≥0.30


,prob_threshold,precision,recall,f1,coverage,n_queries,n_predictions,n_truepos
0,0.30,0.444444,0.4,0.421053,0.9,10,9,4
1,0.35,0.428571,0.3,0.352941,0.7,10,7,3
2,0.40,0.428571,0.3,0.352941,0.7,10,7,3
3,0.45,0.428571,0.3,0.352941,0.7,10,7,3
4,0.50,0.428571,0.3,0.352941,0.7,10,7,3
5,0.55,0.000000,0.0,0.000000,0.3,10,3,0
6,0.60,0.000000,0.0,0.000000,0.3,10,3,0
7,0.65,0.000000,0.0,0.000000,0.3,10,3,0
8,0.70,0.000000,0.0,0.000000,0.2,10,2,0
9,0.75,0.000000,0.0,0.000000,0.2,10,2,0


## G) Reporte & Análisis de errores

In [40]:
# --- G) Informe final: modelo ganador, umbral y análisis de errores (Sprint 1) -------------------
import os
import re
import numpy as np
import pandas as pd

# Rutas por defecto (coherentes con E/F)
BASE        = os.getcwd()
RESULTS_DIR = os.path.join(BASE, "results")

In [41]:
def _load_if_missing(var_name: str, filename: str) -> pd.DataFrame:
    """
    Si existe el DataFrame en memoria (globals), lo devuelve.
    Si no, intenta leerlo de results/<filename>. Lanza un error claro si no existe.
    """
    if var_name in globals() and isinstance(globals()[var_name], pd.DataFrame):
        return globals()[var_name]
    path = os.path.join(RESULTS_DIR, filename)
    assert os.path.isfile(path), f"No encuentro {path}. Asegúrate de haber ejecutado F o de tener los CSV."
    return pd.read_csv(path)

In [42]:
# 1) Cargar/recuperar tablas clave
RANKING_SUMMARY = _load_if_missing("RANKING_SUMMARY", "eval_ranking_summary.csv")
DF_THR          = _load_if_missing("DF_THR",          "eval_thresholds.csv")
BEST_THR        = _load_if_missing("BEST_THR",        "eval_best_thresholds.csv")

In [43]:
# 2) Seleccionar modelo ganador y umbral recomendado (robusto)
def _select_winner_model(ranking_summary: pd.DataFrame) -> str:
    tmp = ranking_summary.copy()
    # Orden: nDCG@10 desc, desempate Recall@1 desc
    sort_cols = []
    if "nDCG@10" in tmp.columns: sort_cols.append("nDCG@10")
    if "Recall@1" in tmp.columns: sort_cols.append("Recall@1")
    assert sort_cols, "RANKING_SUMMARY no contiene 'nDCG@10' ni 'Recall@1'."
    tmp = tmp.sort_values(sort_cols, ascending=[False]*len(sort_cols))
    return str(tmp.iloc[0]["model"])

In [44]:
if "WINNER" in globals():
    WINNER_MODEL = str(WINNER)
else:
    WINNER_MODEL = _select_winner_model(RANKING_SUMMARY)

if "RECOMMENDED_THR" in globals() and isinstance(RECOMMENDED_THR, (float, int)):
    WINNER_THR = float(RECOMMENDED_THR)
else:
    if WINNER_MODEL in BEST_THR["model"].values:
        WINNER_THR = float(BEST_THR[BEST_THR["model"] == WINNER_MODEL].iloc[0]["threshold"])
    else:
        # Fallback: si no hay entry en BEST_THR, usa el mejor f1 global del ganador
        tmp = DF_THR[DF_THR["model"] == WINNER_MODEL].sort_values(["f1","coverage"], ascending=[False, False])
        assert not tmp.empty, "No hay datos de thresholds para el modelo ganador."
        WINNER_THR = float(tmp.iloc[0]["threshold"])

# 3) Resumen ejecutivo del ganador
if WINNER_MODEL in RANKING_SUMMARY["model"].values:
    row_sum = RANKING_SUMMARY[RANKING_SUMMARY["model"] == WINNER_MODEL].iloc[0]
    ndcg10  = float(row_sum.get("nDCG@10", np.nan))
    mrr     = float(row_sum.get("MRR", np.nan))
    r1      = float(row_sum.get("Recall@1", np.nan))
else:
    ndcg10 = mrr = r1 = np.nan

# Métricas a umbral recomendado (precision/recall/f1/coverage)
row_thr = DF_THR[(DF_THR["model"] == WINNER_MODEL) & (DF_THR["threshold"] == WINNER_THR)]
if row_thr.empty:
    # si no hay exact match, cogemos el más cercano
    df_win = DF_THR[DF_THR["model"] == WINNER_MODEL].copy()
    df_win["thr_diff"] = (df_win["threshold"] - WINNER_THR).abs()
    row_thr = df_win.sort_values("thr_diff").head(1)
prec = float(row_thr.iloc[0]["precision"])
rec  = float(row_thr.iloc[0]["recall"])
f1   = float(row_thr.iloc[0]["f1"])
cov  = float(row_thr.iloc[0]["coverage"])

print("=== INFORME DEL GANADOR ===")
print(f"Modelo ganador:        {WINNER_MODEL}")
print(f"Umbral recomendado:    {WINNER_THR:.3f}  |  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  Coverage={cov:.3f}")
print(f"Métricas de ranking:   nDCG@10={ndcg10:.3f}  MRR={mrr:.3f}  Recall@1={r1:.3f}")

# 4) Tabla compacta de ranking (ordenada)
try:
    display(RANKING_SUMMARY.sort_values(["nDCG@10","Recall@1"], ascending=[False, False]))
except Exception:
    pass

# 5) Barrido alrededor del umbral recomendado (±0.02)
win_sweep = DF_THR[DF_THR["model"] == WINNER_MODEL].copy()
win_sweep["dist"] = (win_sweep["threshold"] - WINNER_THR).abs()
near = win_sweep[win_sweep["dist"] <= 0.02].sort_values(["dist","f1","coverage"], ascending=[True, False, False])
print("\n--- Barrido alrededor del umbral (±0.02) ---")
try:
    display(near[["model","threshold","precision","recall","f1","coverage"]].head(10))
except Exception:
    print(near[["model","threshold","precision","recall","f1","coverage"]].head(10).to_string(index=False))

=== INFORME DEL GANADOR ===
Modelo ganador:        sentence-transformers/distiluse-base-multilingual-cased-v2
Umbral recomendado:    0.560  |  Precision=0.333  Recall=0.300  F1=0.316  Coverage=0.900
Métricas de ranking:   nDCG@10=0.510  MRR=0.465  Recall@1=0.150


,model,MRR,nDCG@1,Recall@1,Precision@1,nDCG@5,Recall@5,Precision@5,nDCG@10,Recall@10,Precision@10
4,sentence-transformers/distiluse-base-multiling...,0.465202,0.3,0.15,0.3,0.416914,0.45,0.18,0.509734,0.65,0.13
3,sentence-transformers/all-mpnet-base-v2,0.415278,0.2,0.10,0.2,0.360842,0.45,0.18,0.462761,0.70,0.14
1,sentence-transformers/all-MiniLM-L6-v2,0.401458,0.2,0.10,0.2,0.358155,0.45,0.18,0.440732,0.65,0.13
7,sentence-transformers/sentence-t5-base,0.405867,0.2,0.10,0.2,0.353904,0.45,0.18,0.432364,0.65,0.13
0,embedding-data/deberta-sentence-transformer,0.385498,0.3,0.15,0.3,0.354377,0.30,0.12,0.423331,0.40,0.08
6,sentence-transformers/paraphrase-albert-small-v2,0.380078,0.2,0.10,0.2,0.396840,0.45,0.18,0.415297,0.50,0.10
2,sentence-transformers/all-distilroberta-v1,0.357450,0.1,0.05,0.1,0.348530,0.50,0.20,0.411247,0.65,0.13
5,sentence-transformers/paraphrase-MiniLM-L3-v2,0.315972,0.1,0.05,0.1,0.273846,0.40,0.16,0.367533,0.65,0.13



--- Barrido alrededor del umbral (±0.02) ---


,model,threshold,precision,recall,f1,coverage
262,sentence-transformers/distiluse-base-multiling...,0.56,0.333333,0.3,0.315789,0.9
263,sentence-transformers/distiluse-base-multiling...,0.58,0.250000,0.2,0.222222,0.8


In [45]:
# 6) Carga/uso de FP y FN (si no están en memoria, cargarlos de disco)
def _load_errors_for_model(model_name: str):
    safe = _sanitize_for_filename(model_name)
    fp_path = os.path.join(RESULTS_DIR, f"errors_fp_{safe}.csv")
    fn_path = os.path.join(RESULTS_DIR, f"errors_fn_{safe}.csv")
    df_fp = pd.read_csv(fp_path) if os.path.isfile(fp_path) else pd.DataFrame()
    df_fn = pd.read_csv(fn_path) if os.path.isfile(fn_path) else pd.DataFrame()
    return df_fp, df_fn

In [46]:
if "DF_FP" in globals() and "DF_FN" in globals():
    df_fp_show, df_fn_show = DF_FP.copy(), DF_FN.copy()
else:
    df_fp_show, df_fn_show = _load_errors_for_model(WINNER_MODEL)

print("\n--- Muestras de errores (para análisis) ---")
if not df_fp_show.empty:
    print(f"[FP] Falsos positivos con score ≥ {WINNER_THR:.3f} (top 10 por score):")
    try:
        display(df_fp_show.sort_values("top1_score", ascending=False).head(10))
    except Exception:
        print(df_fp_show.sort_values("top1_score", ascending=False).head(10).to_string(index=False))
else:
    print("[FP] No hay archivo/filas de falsos positivos para este modelo/umbral.")

if not df_fn_show.empty:
    print(f"\n[FN] Falsos negativos por abstención (positivo en topK pero score_top1 < {WINNER_THR:.3f}):")
    try:
        display(df_fn_show.sort_values(["bestpos_rank","bestpos_score"], ascending=[True, False]).head(10))
    except Exception:
        print(df_fn_show.sort_values(["bestpos_rank","bestpos_score"], ascending=[True, False]).head(10).to_string(index=False))
else:
    print("[FN] No hay archivo/filas de falsos negativos para este modelo/umbral.")

# 7) Recomendaciones operativas sencillas
print("\n--- Recomendaciones ---")
print("- Si quieres más PRECISIÓN (menos FP): sube el umbral +0.02 a +0.05 y observa la caída de coverage/recall.")
print("- Si quieres más RECALL (menos FN): baja el umbral -0.02 a -0.05 y limita el top-K en producción.")
print("- Revisa FP/FN por 'delta_top1_top2', 'lenpen', 'stopjac' para ajustar lam_len / rho_stop en el Bloque D.")



--- Muestras de errores (para análisis) ---
[FP] Falsos positivos con score ≥ 0.560 (top 10 por score):


,query_label,top1_label,top1_score,top1_sim,top1_z,delta_top1_top2,top1_lenpen,top1_stopjac,top1_mnn,positives
4,Supervise engineering or other technical perso...,Provide technical guidance to other personnel.,0.857037,0.867269,4.772807,0.146994,0.000000,0.0,1,"[Supervise production or support personnel., S..."
1,Prepare procedural documents.,Prepare proposal documents.,0.838008,0.768583,4.649864,0.045406,0.000000,0.0,1,[Document organizational or operational proced...
3,Estimate operational costs.,Estimate costs of goods or services.,0.790238,0.736054,3.652020,0.312485,0.500000,0.0,1,"[Estimate labor requirements., Estimate time r..."
5,Research engineering aspects of biological or ...,Develop biological research methods.,0.696772,0.712373,3.930448,0.018934,0.500000,0.0,1,[Research engineering applications of emerging...
0,Train personnel on proper operational procedures.,Monitor organizational procedures to ensure pr...,0.660958,0.525858,2.809335,0.213245,0.142857,0.0,1,[Instruct college students in physical or life...
2,Develop operational methods or processes that ...,Develop technical methods or processes.,0.651843,0.580014,2.707367,0.093016,0.583333,0.5,1,[Implement transportation changes to reduce en...


[FN] No hay archivo/filas de falsos negativos para este modelo/umbral.

--- Recomendaciones ---
- Si quieres más PRECISIÓN (menos FP): sube el umbral +0.02 a +0.05 y observa la caída de coverage/recall.
- Si quieres más RECALL (menos FN): baja el umbral -0.02 a -0.05 y limita el top-K en producción.
- Revisa FP/FN por 'delta_top1_top2', 'lenpen', 'stopjac' para ajustar lam_len / rho_stop en el Bloque D.


## H) Calibración visual: curvas de precisión, recall, F1 y coverage

In [50]:
import os, re
import numpy as np
import pandas as pd

# Plotly (sin binarios); set renderer robusto
try:
    import plotly.express as px
    import plotly.io as pio
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly>=5.20"])
    import plotly.express as px
    import plotly.io as pio

# Forzar un renderer que funciona en la mayoría de notebooks
# Puedes cambiar a 'vscode' si usas VS Code Notebook, o 'colab' si estás en Colab.
try:
    pio.renderers.default = "notebook_connected"
except Exception:
    # Fallbacks razonables
    for r in ("vscode", "jupyterlab", "colab", "iframe_connected", "browser"):
        try:
            pio.renderers.default = r
            break
        except Exception:
            continue

RESULTS_DIR = os.path.join(os.getcwd(), "results")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

In [51]:
def _sanitize(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "__", str(name)).strip("._-")

def _exists_df(name: str) -> bool:
    return name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0

def _load_csv(path: str) -> pd.DataFrame:
    if not os.path.isfile(path):
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def _collect_models_from_known():
    models = set()
    if _exists_df("DF_THR"): 
        models |= set(pd.Series(globals()["DF_THR"]["model"]).astype(str).unique())
    if _exists_df("BEST_THR"):
        models |= set(pd.Series(globals()["BEST_THR"]["model"]).astype(str).unique())
    if _exists_df("RANKING_SUMMARY"):
        models |= set(pd.Series(globals()["RANKING_SUMMARY"]["model"]).astype(str).unique())
    return sorted(models)

def _build_plot_table() -> pd.DataFrame:
    """
    Devuelve DataFrame con columnas:
      ['model','prob_threshold','precision','recall','f1','coverage']
    Prioridad:
      1) CAL_RESULTS en memoria
      2) cal_sweep_<modelo>.csv en results/
      3) DF_THR en memoria (renombrando 'threshold'->'prob_threshold')
      4) results/eval_thresholds.csv
    """
    # 1) Desde CAL_RESULTS (memoria, calibrado)
    if "CAL_RESULTS" in globals() and isinstance(CAL_RESULTS, dict) and len(CAL_RESULTS) > 0:
        frames = []
        for m, obj in CAL_RESULTS.items():
            df_sw = obj.get("sweep", pd.DataFrame()).copy()
            if df_sw.empty:
                continue
            if "prob_threshold" not in df_sw.columns and "threshold" in df_sw.columns:
                df_sw = df_sw.rename(columns={"threshold": "prob_threshold"})
            df_sw["model"] = str(m)
            frames.append(df_sw[["model","prob_threshold","precision","recall","f1","coverage"]])
        if frames:
            return pd.concat(frames, ignore_index=True)

    # 2) Disco: cal_sweep_<modelo>.csv
    frames = []
    models = _collect_models_from_known()
    disk_files = [f for f in os.listdir(RESULTS_DIR) if f.startswith("cal_sweep_") and f.endswith(".csv")] if os.path.isdir(RESULTS_DIR) else []
    if disk_files and not models:
        models = [f[len("cal_sweep_"):-4] for f in disk_files]
    for m in models:
        path = os.path.join(RESULTS_DIR, f"cal_sweep_{_sanitize(m)}.csv")
        df_sw = _load_csv(path)
        if df_sw.empty:
            continue
        if "prob_threshold" not in df_sw.columns and "threshold" in df_sw.columns:
            df_sw = df_sw.rename(columns={"threshold": "prob_threshold"})
        df_sw["model"] = str(m)
        frames.append(df_sw[["model","prob_threshold","precision","recall","f1","coverage"]])
    if frames:
        return pd.concat(frames, ignore_index=True)

    # 3) DF_THR (no calibrado) en memoria
    if _exists_df("DF_THR"):
        df = globals()["DF_THR"].copy()
        df = df.rename(columns={"threshold": "prob_threshold"})
        cols = ["model","prob_threshold","precision","recall","f1","coverage"]
        if set(cols).issubset(df.columns):
            return df[cols].copy()

    # 4) eval_thresholds.csv en disco
    df = _load_csv(os.path.join(RESULTS_DIR, "eval_thresholds.csv"))
    if not df.empty:
        if "prob_threshold" not in df.columns and "threshold" in df.columns:
            df = df.rename(columns={"threshold": "prob_threshold"})
        cols = ["model","prob_threshold","precision","recall","f1","coverage"]
        if set(cols).issubset(df.columns):
            return df[cols].copy()

    return pd.DataFrame(columns=["model","prob_threshold","precision","recall","f1","coverage"])

def _show_and_save(fig, filename_hint: str):
    """Muestra el fig y además lo guarda como HTML en results/plots/."""
    safe = _sanitize(filename_hint)
    html_path = os.path.join(PLOTS_DIR, f"{safe}.html")
    try:
        fig.show()
    except Exception as e:
        print(f"[H] Aviso: fig.show() no pudo renderizar ({e}). Guardando HTML.")
    try:
        fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
        print(f"[H] Guardado: {html_path}")
    except Exception as e:
        print(f"[H] No se pudo guardar HTML: {e}")

def _plot_model_curves(thr_summary: pd.DataFrame, model_name: str):
    """Curvas de precisión, recall, F1 y coverage vs umbral (por modelo)."""
    df = thr_summary[thr_summary["model"] == model_name].copy()
    if df.empty:
        print(f"[H] No hay datos para {model_name}")
        return
    df = df.sort_values("prob_threshold")
    curves = df.melt(id_vars=["prob_threshold"],
                     value_vars=["precision","recall","f1","coverage"],
                     var_name="metric", value_name="value")
    fig = px.line(curves, x="prob_threshold", y="value", color="metric", markers=True,
                  title=f"Curvas de calibración — {model_name}",
                  labels={"prob_threshold":"Umbral (prob o score norm.)","value":"Métrica"})
    # Mejor F1
    if not df["f1"].isna().all():
        idx = int(df["f1"].idxmax())
        best_thr = float(df.loc[idx, "prob_threshold"])
        best_f1  = float(df.loc[idx, "f1"])
        fig.add_vline(x=best_thr, line_dash="dash", opacity=0.6)
        fig.add_annotation(x=best_thr, y=best_f1,
                           text=f"F1 máx={best_f1:.3f} @ ≥{best_thr:.2f}",
                           showarrow=True, yshift=10)
    _show_and_save(fig, f"calibration_curves__{model_name}")

def _plot_all_models_f1(thr_summary: pd.DataFrame):
    """Comparativa: F1 vs umbral para todos los modelos."""
    if thr_summary.empty:
        print("[H] No hay datos para graficar.")
        return
    df = thr_summary.copy().sort_values(["model","prob_threshold"])
    fig = px.line(df, x="prob_threshold", y="f1", color="model", markers=True,
                  title="Comparativa F1 vs umbral (todos los modelos)",
                  labels={"prob_threshold":"Umbral (prob o score norm.)","f1":"F1"})
    _show_and_save(fig, "calibration_f1_all_models")

In [52]:
# Construir tabla (prioriza calibrado; si no hay, usa crudo)
THR_SUMMARY = _build_plot_table()
if THR_SUMMARY.empty:
    raise RuntimeError("[H] No hay sweeps (calibrados ni crudos). Ejecuta el Bloque F primero o revisa 'results/'.")

# Graficar por modelo
for model_name in THR_SUMMARY["model"].astype(str).unique():
    _plot_model_curves(THR_SUMMARY, model_name)

# Comparativa global
_plot_all_models_f1(THR_SUMMARY)

print(f"[H] Renderer por defecto: {pio.renderers.default}")
print(f"[H] Si no ves los gráficos inline, abre los HTML en: {PLOTS_DIR}")
# --- Fin Bloque H (v2) ---------------------------------------------------------------------------

[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__embedding-data__deberta-sentence-transformer.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__all-MiniLM-L6-v2.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__all-distilroberta-v1.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__all-mpnet-base-v2.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__distiluse-base-multilingual-cased-v2.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__paraphrase-MiniLM-L3-v2.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__paraphrase-albert-small-v2.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_curves__sentence-transformers__sentence-t5-base.html


[H] Guardado: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots/calibration_f1_all_models.html
[H] Renderer por defecto: notebook_connected
[H] Si no ves los gráficos inline, abre los HTML en: /home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/results/plots
